In [1]:
import torch
import numpy as np
import json
from tqdm import tqdm
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer
from collections import Counter
import os

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\KamilSarzyniak\anaconda3\envs\AI_DL_ML\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
print(torch.cuda.device_count())
print(torch.cuda.get_device_name())

1
NVIDIA GeForce RTX 3060 Laptop GPU


In [3]:
with open("course_goals.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [4]:
course_goals_texts = [example["course_goals"] for example in data if "course_goals" in example]

In [6]:
model, tokenizer = FastLanguageModel.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_seq_length=1024,
    dtype=torch.float32,
    load_in_4bit=True,
)

==((====))==  Unsloth 2025.1.8: Fast Llama patching. Transformers: 4.48.2.
   \\   /|    GPU: NVIDIA GeForce RTX 3060 Laptop GPU. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.6.0+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [7]:
def get_embedding(text, model, tokenizer, max_length=512):
    """Tworzy embedding dla podanego tekstu przy użyciu modelu."""
    input_ids = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=max_length).input_ids.to(model.device)
    with torch.no_grad():
        output = model(input_ids, output_hidden_states=True)
    embedding = output.hidden_states[-1].mean(dim=1).cpu().numpy()
    return embedding

In [8]:
embedding_file = "course_goals_embeddings.npy"

In [14]:
if os.path.exists(embedding_file):
    print("Ładowanie zapisanych embeddingów...")
    course_goals_embeddings = np.load(embedding_file)
else:
    print("Generowanie embeddingów dla kursów...")

    course_goals_embeddings = np.vstack(
        [get_embedding(text, model, tokenizer) for text in tqdm(course_goals_texts, desc="Postęp", unit="kurs")]
    )

    course_goals_embeddings = normalize(course_goals_embeddings)

    np.save(embedding_file, course_goals_embeddings)
    print(f"Embeddingi zapisane do pliku: {embedding_file}")

Generowanie embeddingów dla kursów...


RuntimeError: Failed to find C compiler. Please specify via CC environment variable.

In [49]:
user_query = input("Podaj opis kursu, który Cię interesuje: ")

user_embedding = get_embedding(user_query)

similarities = cosine_similarity(user_embedding, course_goals_embeddings)[0]

best_match_idx = np.argmax(similarities)
best_match_text = course_goals_texts[best_match_idx]

print("\n Najbardziej dopasowany kurs:")
print(best_match_text)

🔄 Generowanie embeddingów dla kursów...


Postęp:   0%|          | 2/631 [00:06<32:39,  3.12s/kurs]

KeyboardInterrupt: 